# Dependencies

In [1]:
from hipt_4k import HIPT_4K
from hipt_model_utils import get_vit256, get_vit4k, eval_transforms
from hipt_heatmap_utils import *
light_jet = cmap_map(lambda x: x/2 + 0.5, matplotlib.cm.jet)

pretrained_weights256 = '../Checkpoints/vit256_small_dino.pth'
pretrained_weights4k = '../Checkpoints/vit4k_xs_dino.pth'
device256 = torch.device('cpu')
device4k = torch.device('cpu')

### ViT_256 + ViT_4K loaded independently (used for Attention Heatmaps)
model256 = get_vit256(pretrained_weights=pretrained_weights256, device=device256)
model4k = get_vit4k(pretrained_weights=pretrained_weights4k, device=device4k)

### ViT_256 + ViT_4K loaded into HIPT_4K API
model = HIPT_4K(pretrained_weights256, pretrained_weights4k, device256, device4k)
model.eval()

# of Patches: 196
# of Patches: 196


HIPT_4K(
  (model256): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (blocks): ModuleList(
      (0): Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (drop_path): Identity()
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU()
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )
      (1): Block(
        (norm1): LayerNorm((384,),

# Standalone HIPT_4K Model Inference

In [2]:
region = Image.open('./image_demo/image_4k.png')
x = eval_transforms()(region).unsqueeze(dim=0)
print('Input Shape:', x.shape)
print('Output Shape:', model.forward(x).shape)

Input Shape: torch.Size([1, 3, 4096, 4096])


/sibcb1/chenluonanlab8/zoujiawei/miniconda3/envs/CarfHe/lib/python3.9/site-packages/torch/nn/functional.py:3060: UserWarning: Default upsampling behavior when mode=bicubic is changed to align_corners=False since 0.4.0. Please specify align_corners=True if the old behavior is desired. See the documentation of nn.Upsample for details.
  warnings.warn("Default upsampling behavior when mode={} is changed "
/sibcb1/chenluonanlab8/zoujiawei/miniconda3/envs/CarfHe/lib/python3.9/site-packages/torch/nn/functional.py:3103: UserWarning: The default behavior for interpolate/upsample with float scale_factor changed in 1.6.0 to align with other frameworks/libraries, and now uses scale_factor directly, instead of relying on the computed output size. If you wish to restore the old behavior, please set recompute_scale_factor=True. See the documentation of nn.Upsample for details. 
  warnings.warn("The default behavior for interpolate/upsample with float scale_factor changed "


Output Shape: torch.Size([1, 192])


# HIPT_4K Attention Heatmaps
Code for producing attention results (for [256 x 256], [4096 x 4096], and hierarchical [4096 x 4096]) can be run (as-is) below. There are several ways these results can be run:
1. **hipt_4k.py** Class (Preferred): This class blends inference and heatmap creation in a seamless and more object-oriented manner, and is where I am focusing my future code development around.
2. Helper Functions in **hipt_heatmap_utils.py** (Soon-to-be-deprecated): Heatmap creation was originally written as helper functions. May be more useful and easier from research perspective.

Please use whatever is most helpful for your use case :) 

### 256 x 256 Demo (Saving Attention Maps Individually)

In [3]:
patch = Image.open('./image_demo/image_256.png')
output_dir = './attention_demo/256_output_indiv/'
os.makedirs(output_dir, exist_ok=True)

def create_patch_heatmaps_mean(patch, model256, output_dir, fname, offset=16, alpha=0.5, 
                                cmap=plt.get_cmap('coolwarm'), device256=torch.device('cpu')):
    r"""
    Creates patch heatmaps by averaging the attention scores across all heads and saves the result.
    
    Args:
    - patch (PIL.Image):        256 x 256 Image 
    - model256 (torch.nn):      256-Level ViT 
    - output_dir (str):         Save directory / subdirectory
    - fname (str):              Naming structure of files
    - offset (int):             How much to offset (from top-left corner with zero-padding) the region by for blending 
    - alpha (float):            Image blending factor for cv2.addWeighted
    - cmap (matplotlib.pyplot): Colormap for creating heatmaps
    
    Returns:
    - None
    """
    # Prepare the patch images
    patch1 = patch.copy()
    patch2 = add_margin(patch.crop((16,16,256,256)), top=0, left=0, bottom=16, right=16, color=(255,255,255))
    
    # Get attention scores for both patches
    _, a256_1 = get_patch_attention_scores(patch1, model256, device256=device256)
    _, a256_2 = get_patch_attention_scores(patch2, model256, device256=device256)
    
    save_region = np.array(patch.copy())
    s = 256
    offset_2 = offset
    
    # Initialize cumulative score array
    cumulative_score256 = np.zeros((s, s))  # 用于累加所有注意力头的得分

    # Calculate and accumulate scores for all attention heads
    for i in range(6):
        score256_1 = get_scores256(a256_1[:, i, :, :], size=(s,) * 2)
        score256_2 = get_scores256(a256_2[:, i, :, :], size=(s,) * 2)
        
        new_score256_2 = np.zeros_like(score256_2)
        new_score256_2[offset_2:s, offset_2:s] = score256_2[:(s - offset_2), :(s - offset_2)]
        
        overlay256 = np.ones_like(score256_2) * 100
        overlay256[offset_2:s, offset_2:s] += 100
        
        score256 = (score256_1 + new_score256_2) / overlay256
        
        cumulative_score256 += score256  # 累加每个注意力头的得分

    # Calculate the average attention score across all heads
    average_score256 = cumulative_score256 / 6

    # Generate and save the heatmap

    # 创建颜色映射
    color_block256 = np.zeros((s, s, 3), dtype=np.uint8)  # 初始化为黑色
    
    # 设置条件颜色映射：<= 0.5 -> 灰色 (128, 128, 128)， > 0.5 -> 黄色 (255, 255, 0)
    color_block256[average_score256 <= 0.5] = [128, 128, 128]  # 灰色
    color_block256[average_score256 > 0.5] = [255, 255, 0]     # 黄色
    region256_hm = cv2.addWeighted(color_block256, alpha, save_region.copy(), 1 - alpha, 0, save_region.copy())
    
    # Save the result
    Image.fromarray(region256_hm).save(os.path.join(output_dir, f'{fname}_256_1avg.tif'))


create_patch_heatmaps_mean(patch=patch, model256=model256, 
                            output_dir=output_dir, fname='patch',
                            cmap=light_jet, device256=device256)

/sibcb1/chenluonanlab8/zoujiawei/miniconda3/envs/CarfHe/lib/python3.9/site-packages/torch/nn/functional.py:3060: UserWarning: Default upsampling behavior when mode=bilinear is changed to align_corners=False since 0.4.0. Please specify align_corners=True if the old behavior is desired. See the documentation of nn.Upsample for details.
  warnings.warn("Default upsampling behavior when mode={} is changed "


In [5]:
patch.size

(256, 256)

### 256 x 256 Demo (Concatenating + Saving Attention Maps)

In [4]:
patch = Image.open('./image_demo/image_256.png')
output_dir = './attention_demo/256_output_concat/'
os.makedirs(output_dir, exist_ok=True)
create_patch_heatmaps_concat(patch=patch, model256=model256, 
                            output_dir=output_dir, fname='patch',
                            cmap=light_jet, device256=device256)

### 4096 x 4096 Demo (Saving Attention Maps Individually)

In [5]:
region = Image.open('./image_demo/image_4k.png')
output_dir = './attention_demo/4k_output_indiv/'
os.makedirs(output_dir, exist_ok=True)
create_hierarchical_heatmaps_indiv(region, model256, model4k, 
                                   output_dir, fname='region', 
                                   scale=2, threshold=0.5, cmap=light_jet, alpha=0.5,
                                   device256=device256, device4k=device4k)

### 4096 x 4096 Demo (Concatenating + Saving Attention Maps)

In [6]:
region = Image.open('./image_demo/image_4k.png')
output_dir = './attention_demo/4k_output_concat/'
os.makedirs(output_dir, exist_ok=True)
create_hierarchical_heatmaps_concat(region, model256, model4k, 
                                   output_dir, fname='region', 
                                   scale=2, cmap=light_jet, alpha=0.5,
                                   device256=device256, device4k=device4k)